In [2]:
%load_ext cudf.pandas

In [3]:
import numpy as np
import pandas as pd

import warnings
warnings.filterwarnings('ignore')

In [4]:
df = pd.read_csv('/kaggle/input/datasets/ashura369/imdb-dataset/IMDB Dataset.csv', engine='python', on_bad_lines='skip')
df.head(10)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
5,"Probably my all-time favorite movie, a story o...",positive
6,I sure would like to see a resurrection of a u...,positive
7,"This show was an amazing, fresh & innovative i...",negative
8,Encouraged by the positive comments about this...,negative
9,If you like original gut wrenching laughter yo...,positive


In [5]:
print(df.duplicated().sum())
df = df.drop_duplicates()
print(df.duplicated().sum())


418
0


In [6]:
!pip install cleantext

In [7]:
import nltk
from nltk.tokenize import word_tokenize

from nltk.corpus import stopwords
stop_words = stopwords.words('english')

from nltk.stem import WordNetLemmatizer
lmt = WordNetLemmatizer()

from cleantext import clean

In [8]:
import re

In [9]:
def transform(txt):
  # removing html tags
  txt = re.sub(r'<[^>]+>', '', txt)

  # lowercase and removing punctuations
  txt = clean(txt, lowercase=True, punct=True)

  # tokenization
  txt = nltk.word_tokenize(txt)

  temp = [lmt.lemmatize(word, pos='v') for word in txt if word not in stop_words]

  temp2 = temp[:]
  temp2 = " ".join(temp2)

  return temp2

In [10]:
transform('Hello this the God King, playing and singing songs of liberation')

'hello god king play sing songs liberation'

In [11]:
df['transformed'] = df['review'].apply(transform)

In [12]:
df.head(5)

,review,sentiment,transformed
0,One of the other reviewers has mentioned that ...,positive,one reviewers mention watch 1 oz episode youll...
1,A wonderful little production. <br /><br />The...,positive,wonderful little production film technique una...
2,I thought this was a wonderful way to spend ti...,positive,think wonderful way spend time hot summer week...
3,Basically there's a family where a little boy ...,negative,basically theres family little boy jake think ...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,petter matteis love time money visually stun f...


## Type 1

In [13]:
import gensim
from gensim.utils import simple_preprocess
from nltk import sent_tokenize     # for the time being we will be using sentence tokenization, instead of word tokenization

### What does Gensim's `simple_preprocess` do?

`simple_preprocess` is an all-in-one text cleaning function designed to instantly prepare text for Word2Vec. In a single step, it automatically:
1. **Lowercases everything** (e.g., "Hello" → "hello")
2. **Removes punctuation and special characters** (removes `.`, `,`, `\n`, etc.)
3. **Word Tokenizes** (splits the sentence into a list of individual words)
4. **Removes very short/long words** (By default, deletes words shorter than 2 characters like "i" or "a", and words longer than 15 characters).
*Note: If you want to keep 1-letter words, you can override the default by using: `simple_preprocess(text, min_len=1)`*

In [14]:
def transform(txt):
    txt = nltk.sent_tokenize(txt)
    temp = []
    for words in txt:
        words = simple_preprocess(words)
        temp.extend(words) 
    
    return temp

In [15]:
sent_tokenize('Hello my name is king. i am from Bhubaneswar Odisha. \n I am a data scientist by profession')

['Hello my name is king.',
 'i am from Bhubaneswar Odisha.',
 'I am a data scientist by profession']

In [16]:
simple_preprocess('HELLO !!! my name is king. i am from Bhubaneswar Odisha. \n I am a data scientist by profession')

['hello',
 'my',
 'name',
 'is',
 'king',
 'am',
 'from',
 'bhubaneswar',
 'odisha',
 'am',
 'data',
 'scientist',
 'by',
 'profession']

In [17]:
transform('Hello my name is king. i am from Bhubaneswar Odisha. \n I am a data scientist by profession')

['hello',
 'my',
 'name',
 'is',
 'king',
 'am',
 'from',
 'bhubaneswar',
 'odisha',
 'am',
 'data',
 'scientist',
 'by',
 'profession']

From above we can see that, it is breaking a review into list different sentences, and then breaking each list of sentences into list of words. 

In [18]:
vocab = df['transformed'].apply(transform)
# MAKE SURE TO RECEIVE THE VOCABULARY IN LIST FORMAT

In [19]:
print(len(vocab))
print(vocab[:10])

49582
0    [one, reviewers, mention, watch, oz, episode, ...
1    [wonderful, little, production, film, techniqu...
2    [think, wonderful, way, spend, time, hot, summ...
3    [basically, theres, family, little, boy, jake,...
4    [petter, matteis, love, time, money, visually,...
5    [probably, alltime, favorite, movie, story, se...
6    [sure, would, like, see, resurrection, date, s...
7    [show, amaze, fresh, innovative, idea, first, ...
8    [encourage, positive, comment, film, look, for...
9    [like, original, gut, wrench, laughter, like, ...
Name: transformed, dtype: list


In [20]:
model = gensim.models.Word2Vec(
    window = 10,
    min_count = 2
)

In [21]:
model.build_vocab(vocab)
model.train(vocab, total_examples=model.corpus_count, epochs=model.epochs)

(27047208, 29332415)

In [25]:
print(len(model.wv.index_to_key))
model.wv.index_to_key[:20]

69869


['film',
 'movie',
 'one',
 'make',
 'like',
 'see',
 'get',
 'time',
 'good',
 'character',
 'watch',
 'go',
 'even',
 'would',
 'think',
 'really',
 'story',
 'show',
 'look',
 'say']

In [28]:
def document_vector(txt):
    doc = [word for word in txt.split() if word in model.wv.index_to_key]

    temp = np.mean(model.wv[doc], axis=0)
    return temp

In [31]:
print(len(document_vector(df['transformed'].values[0])))
print(document_vector(df['transformed'].values[0]))

100
[ 0.25328505  0.39883786 -0.10268155 -0.04061913  0.4079964  -0.7392929
 -0.15362376  0.08470528 -0.50880563 -0.12359331 -0.2866497  -0.52672166
  0.19627707  0.11244937 -0.01781456 -0.711878    0.40584812 -0.13229392
 -0.29696438 -0.30109066  0.2544345   0.01059381  0.00799837 -0.59987247
 -0.08651406  0.26764798 -0.21994509  0.03726386 -0.30938682 -0.27171165
 -0.5266659   0.16765736 -0.10749479  0.07435792  0.01465461 -0.20820233
  0.32252863 -0.78152305 -0.49703476 -0.52521735  0.3389036   0.21094495
  0.47444245 -0.2384705   0.13566004  0.1900956  -0.19939166  0.24480687
  0.0798864   0.35288072  0.30137384 -0.21056966  0.12508342 -0.28426853
  0.31309414 -0.52579045  0.24087843  0.32736263 -0.09322519  0.0714126
 -0.56246924 -0.3662069  -0.04382085  0.15425815 -0.10629948 -0.15285176
  0.14075504  0.35411924 -0.24529901  0.2641074  -0.15183386  0.15840994
  0.47322553  0.6038701  -0.18879558 -0.04264804  0.49724424  0.41919303
 -0.6458108   0.41518733 -0.5780806   0.15996489 

In [32]:
from tqdm import tqdm

In [35]:
x = []
for doc in tqdm(df['transformed'].values):
    x.append(document_vector(doc))

100%|██████████| 49582/49582 [10:20<00:00, 79.91it/s] 


In [37]:
x = np.array(x)
x[0]

array([ 0.25328505,  0.39883786, -0.10268155, -0.04061913,  0.4079964 ,
       -0.7392929 , -0.15362376,  0.08470528, -0.50880563, -0.12359331,
       -0.2866497 , -0.52672166,  0.19627707,  0.11244937, -0.01781456,
       -0.711878  ,  0.40584812, -0.13229392, -0.29696438, -0.30109066,
        0.2544345 ,  0.01059381,  0.00799837, -0.59987247, -0.08651406,
        0.26764798, -0.21994509,  0.03726386, -0.30938682, -0.27171165,
       -0.5266659 ,  0.16765736, -0.10749479,  0.07435792,  0.01465461,
       -0.20820233,  0.32252863, -0.78152305, -0.49703476, -0.52521735,
        0.3389036 ,  0.21094495,  0.47444245, -0.2384705 ,  0.13566004,
        0.1900956 , -0.19939166,  0.24480687,  0.0798864 ,  0.35288072,
        0.30137384, -0.21056966,  0.12508342, -0.28426853,  0.31309414,
       -0.52579045,  0.24087843,  0.32736263, -0.09322519,  0.0714126 ,
       -0.56246924, -0.3662069 , -0.04382085,  0.15425815, -0.10629948,
       -0.15285176,  0.14075504,  0.35411924, -0.24529901,  0.26

In [39]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()

In [40]:
y = encoder.fit_transform(df['sentiment'])
y

array([1, 1, 1, ..., 0, 0, 0])

In [41]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [43]:
X_train,X_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=1)

In [45]:
rf = RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)

In [46]:
from sklearn.metrics import confusion_matrix

In [47]:
print(f"ACCURACY SCORE : {accuracy_score(y_test, y_pred)}")
print(f"\n CONFUSION MATRIX : \n{confusion_matrix(y_test, y_pred)}")

ACCURACY SCORE : 0.8388625592417062

 CONFUSION MATRIX : 
[[4115  918]
 [ 680 4204]]
